In [ ]:
!pip install groq chromadb sentence-transformers pandas -q

In [ ]:
import pandas as pd
import sqlite3
from groq import Groq
import chromadb
import os
import json

print("All libraries are imported successfully !")
print("Libraries loaded : Pandas, sqlite3, chromadb")

All libraries are imported successfully !
Libraries loaded : Pandas, sqlite3, chromadb


In [ ]:
GROQ_API_KEY = "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"
client = Groq(api_key = GROQ_API_KEY)

MODEL = "llama-3.1-8b-instant"
print("Groq client configured")
print(f"Model : {MODEL}")
print("Status : Ready to generate AI responses")

Groq client configured
Model : llama-3.1-8b-instant
Status : Ready to generate AI responses


In [ ]:
responsible_system_prompt = """
you are a helpful data analysis assistant.
You only answer questions based on the data provided to you.
If you are not sure about searching , say : I do not have enough information to answer that accurately.
Never make up statistics or facts that are not in the data you receive.
"""

test_response = client.chat.completions.create(
    model = MODEL,
    max_tokens = 200,
    messages = [
        {
            "role" : "system",
            "content" : responsible_system_prompt
        },
        {
            "role" : "user",
            "content" : "What is the population in mars?"
        }
    ]
)

ai_answer = test_response.choices[0].message.content
print("==== Responsible AI Test ====")
print("Question : What is the population of mars ?")
print("Answer : ",ai_answer)
print("\nNote : A responsible AI should admit it cannot ")

==== Responsible AI Test ====
Question : What is the population of mars ?
Answer :  I do not have enough information to answer that accurately.

Note : A responsible AI should admit it cannot 


In [ ]:
student_df = pd.read_csv("student_performance (1).csv")

print("=== Student Performance Data ===")
print(f"Shape : {student_df.shape[0]} students , {student_df.shape[1]} columns")
print()
print(student_df.head(5))

=== Student Performance Data ===
Shape : 30 students , 11 columns

   student_id          name  age  gender branch  attendance_pct  \
0           1  Aarav Sharma   20    Male    CSE              85   
1           2   Priya Patel   21  Female    ECE              92   
2           3   Rohit Kumar   20    Male   MECH              67   
3           4    Sneha Iyer   22  Female    CSE              95   
4           5  Vikram Singh   21    Male  CIVIL              72   

   assignment_score  midterm_score  final_score  gpa passed  
0                78             72           76  7.6    Yes  
1                88             85           89  8.9    Yes  
2                55             60           58  5.8    Yes  
3                92             90           94  9.4    Yes  
4                62             65           63  6.3    Yes  


In [ ]:
notes_df = pd.read_csv("college_notes (1).csv")
print("=== College Performance Dataset ===")
print(f"Shape:{notes_df.shape[0]} notes,{notes_df.shape[1]} columns")
print()
print(notes_df[['note_id','subject','topic','difficulty']].to_string(index=False))

=== College Performance Dataset ===
Shape:15 notes,6 columns

 note_id             subject                       topic   difficulty
       1     Data Structures                      Arrays     Beginner
       2     Data Structures                Linked Lists     Beginner
       3     Data Structures                Binary Trees Intermediate
       4     Data Structures           Stacks and Queues     Beginner
       5 Database Management                  SQL Basics     Beginner
       6 Database Management               Normalization Intermediate
       7 Database Management                    Indexing Intermediate
       8    Machine Learning                  Regression Intermediate
       9    Machine Learning              Classification Intermediate
      10    Machine Learning                  Clustering     Advanced
      11  Python Programming                   Functions     Beginner
      12  Python Programming Object Oriented Programming Intermediate
      13  Python Programming

In [ ]:
print("=== Data Quality Report:student_performance.csv ===")
print("Missing values per column:")
print(student_df.isnull().sum())
print()
print("Data Types:")
print(student_df.dtypes)
print()
print(f"Duplicate rows:{student_df.duplicated().sum()}")
print("Data quality check complete:")

=== Data Quality Report:student_performance.csv ===
Missing values per column:
student_id          0
name                0
age                 0
gender              0
branch              0
attendance_pct      0
assignment_score    0
midterm_score       0
final_score         0
gpa                 0
passed              0
dtype: int64

Data Types:
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object

Duplicate rows:0
Data quality check complete:


In [ ]:
conn = sqlite3.connect(':memory:')
student_df.to_sql('students',conn,if_exists='replace',index=False)
print("SQL database created.")
print("Table'students' loaded with",len(student_df),"rows ,")

SQL database created.
Table'students' loaded with 30 rows ,


In [ ]:
query1 = """
SELECT
    branch,
    COUNT(*) AS total_students,
    ROUND(AVG(gpa), 2) AS avg_gpa,
    ROUND(AVG(attendance_pct), 2) AS avg_attendance
FROM students
GROUP BY branch
ORDER BY avg_gpa DESC;
"""

branch_analysis = pd.read_sql(query1,conn)
print("=== Branch Analysis ===")
print(branch_analysis.to_string(index=False))

=== Branch Analysis ===
branch  total_students  avg_gpa  avg_attendance
    IT               5     8.64           89.40
   CSE              10     7.42           80.00
  MECH               5     7.22           79.40
 CIVIL               4     6.75           75.00
   ECE               6     6.38           69.17


In [ ]:
query3="""
SELECT
  passed,
  COUNT(*) AS student_count,
  ROUND(AVG(gpa),2) AS avg_gpa
FROM students
GROUP BY passed"""

pass_fail=pd.read_sql(query3, conn)
print("=== Pass/Fail Statistics ===")
print(pass_fail.to_string(index=False))

total = len(student_df)
passed = len(student_df[student_df['passed']=='Yes'])
pass_rate = round((passed / total) * 100 , 1)
print(f"\nOverall Pass Rate: {pass_rate}% ({passed}/{total} students)")

=== Pass/Fail Statistics ===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61

Overall Pass Rate: 90.0% (27/30 students)


insight->a particular thing that is taken from a data

In [ ]:
import pandas as pd
query4= """
SELECT
    branch,
    COUNT(*) AS total_students,
    ROUND(AVG(gpa),2) AS avg_gpa,
    ROUND(AVG(attendance_pct),1) AS avg_attendance
FROM students
GROUP BY branch
ORDER BY total_students DESC;
"""

branch_analysis = pd.read_sql(query4, conn)

print(branch_analysis)

  branch  total_students  avg_gpa  avg_attendance
0    CSE              10     7.42            80.0
1    ECE               6     6.38            69.2
2   MECH               5     7.22            79.4
3     IT               5     8.64            89.4
4  CIVIL               4     6.75            75.0


In [ ]:
query5 = """
SELECT
    name,
    branch,
    gpa,
    attendance_pct,
    passed
FROM students
ORDER BY gpa DESC
LIMIT 5;
"""

top_students = pd.read_sql(query5, conn)

print(top_students)

               name branch  gpa  attendance_pct passed
0    Meera Krishnan     IT  9.5              96    Yes
1        Sneha Iyer    CSE  9.4              95    Yes
2  Lakshmi Chandran    CSE  9.2              94    Yes
3      Swathi Menon     IT  9.1              93    Yes
4       Priya Patel    ECE  8.9              92    Yes


In [ ]:
branch_summary = ""

for _, row in branch_analysis.iterrows():
    # iterrows(): loops through each row of a DataFrame
    # row['column_name']: accesses a specific value in that row

    branch_summary += (
        f" - {row['branch']}: {row[...]}"
    )

# Build top students summary

top_summary = ""

for _, row in top_students.iterrows():
    top_summary += (
        f" - {row['name']} ({row['branch']}): GPA {row['gpa']}\n"
    )

# Assemble the complete data summary

data_summary = f"""
STUDENT PERFORMANCE DATA SUMMARY

Total Students: {total}
Overall Pass Rate: {pass_rate}%
Average GPA across all students: {round(student_df['gpa'].mean(), 2)}

Performance by branch:

{branch_summary}

TOP STUDENTS

{top_summary}
"""
print("=== Data Summary ===")
print(data_summary)

=== Data Summary ===

STUDENT PERFORMANCE DATA SUMMARY

Total Students: 30
Overall Pass Rate: 90.0%
Average GPA across all students: 7.29

Performance by branch:

 - CSE: branch             CSE
total_students      10
avg_gpa           7.42
avg_attendance    80.0
Name: 0, dtype: object - ECE: branch             ECE
total_students       6
avg_gpa           6.38
avg_attendance    69.2
Name: 1, dtype: object - MECH: branch            MECH
total_students       5
avg_gpa           7.22
avg_attendance    79.4
Name: 2, dtype: object - IT: branch              IT
total_students       5
avg_gpa           8.64
avg_attendance    89.4
Name: 3, dtype: object - CIVIL: branch            CIVIL
total_students        4
avg_gpa            6.75
avg_attendance     75.0
Name: 4, dtype: object

TOP STUDENTS

 - Meera Krishnan (IT): GPA 9.5
 - Sneha Iyer (CSE): GPA 9.4
 - Lakshmi Chandran (CSE): GPA 9.2
 - Swathi Menon (IT): GPA 9.1
 - Priya Patel (ECE): GPA 8.9




In [ ]:
system_prompt = """
You are an expert academic data analyst working for an engineering college.

You receive student performance summaries and provide clear, actionable insights.

Only use the information provided in the summary.
Do not make up statistics or facts.
"""

user_message = f"""
Here is the student performance data for this semester:

{data_summary}

Please provide:
1. Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""

response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_message
        }
    ]
)

ai_analysis = response.choices[0].message.content

print("=" * 60)
print("AI-Generated Data Analysis")
print("=" * 60)
print(ai_analysis)

AI-Generated Data Analysis
Based on the provided student performance data, here are the insights and recommendations:

1. Key Insights:
   - The IT branch has the highest average GPA (8.64) and the highest Top Students' GPAs with Meera Krishnan and Swathi Menon securing the top two spots. This suggests that the IT department is excelling in terms of student performance.
   - The overall pass rate is 90.0%, indicating a high level of student success across all branches.
   - There appears to be a significant gap in attendance between the branches with the highest and lowest attendance rates (89.4% for IT and 69.2% for ECE).

2. Branch that needs the most improvement:
Based on the data, the ECE branch appears to be the one that needs the most improvement. With a lower average GPA (6.38) and the lowest attendance rate (69.2%), the ECE department should focus on strategies to enhance student performance and engagement.

3. Recommendation for the college principal:
Consider implementing a t